In [2]:
# --- Expense Tracker (Pandas version) ---
import os, csv
import pandas as pd
from datetime import datetime

# -------------------------------
# 1) STATE
# -------------------------------
DATA_FILE = "expenses.csv"
COLS = ["date", "category", "amount", "description"]

# Start with an empty DataFrame
df = pd.DataFrame(columns=COLS)   # date (str YYYY-MM-DD), category (str), amount (float), description (str)
budget = None                     # monthly budget (float or None)

# -------------------------------
# 2) CORE HELPERS
# -------------------------------
def make_expense(date_str, category, amount, description):
    """Validate inputs and return a normalized dict for a single expense."""
    # Date
    try:
        dt = datetime.strptime(str(date_str).strip(), "%Y-%m-%d")
    except ValueError:
        raise ValueError("Date must be YYYY-MM-DD (e.g., 2024-09-18).")
    date_out = dt.strftime("%Y-%m-%d")

    # Category
    category = str(category).strip()
    if not category:
        raise ValueError("Category is required.")

    # Amount
    try:
        amt = float(amount)
    except (TypeError, ValueError):
        raise ValueError("Amount must be a number (e.g., 15.50).")
    if amt < 0:
        raise ValueError("Amount must be non-negative.")

    # Description
    description = str(description).strip()
    if not description:
        raise ValueError("Description is required.")

    return {"date": date_out, "category": category.title(), "amount": round(amt, 2), "description": description}

def clean_df(frame):
    """
    Return a cleaned copy:
    - Keeps only rows with all required columns present/non-null
    - Coerces 'amount' to float and drops negatives or invalids
    - Keeps 'date' as valid YYYY-MM-DD strings
    """
    if frame.empty:
        return frame.copy()

    f = frame.copy()

    # Ensure required columns exist
    for c in COLS:
        if c not in f.columns:
            f[c] = pd.NA

    # Drop rows with any missing required fields
    f = f.dropna(subset=COLS)

    # Validate/normalize amount
    f["amount"] = pd.to_numeric(f["amount"], errors="coerce")
    f = f.dropna(subset=["amount"])
    f = f[f["amount"] >= 0]

    # Validate date format
    def _valid_date(s):
        try:
            datetime.strptime(str(s), "%Y-%m-%d")
            return True
        except Exception:
            return False
    f = f[f["date"].astype(str).map(_valid_date)]

    # Normalize category/description
    f["category"] = f["category"].astype(str).str.strip().str.title()
    f["description"] = f["description"].astype(str).str.strip()

    # Sort by date (nice for display)
    f = f.sort_values("date").reset_index(drop=True)
    return f

def total_spent(frame):
    """Sum of 'amount' over valid rows."""
    f = clean_df(frame)
    return float(f["amount"].sum()) if not f.empty else 0.0

# -------------------------------
# 3) FEATURES (USER ACTIONS)
# -------------------------------
def add_expense_interactive():
    """Prompt for one expense, validate with make_expense, and append to the DataFrame."""
    global df
    while True:
        try:
            date_str = input("Date (YYYY-MM-DD): ").strip()
            category = input("Category (e.g., Food, Travel): ").strip()
            amount = input("Amount (e.g., 15.50): ").strip()
            description = input("Description: ").strip()

            exp = make_expense(date_str, category, amount, description)
            df = pd.concat([df, pd.DataFrame([exp])], ignore_index=True)
            print("✅ Added:", exp)
            break
        except Exception as e:
            print("⚠️", e, "Please try again.\n")

def view_expenses():
    """Show a clean DataFrame of expenses; report if any rows were skipped."""
    f = clean_df(df)
    skipped = len(df) - len(f)
    if f.empty:
        print("No expenses to show.")
        return

    # Nicely formatted display for notebooks
    display_cols = f.copy()
    display_cols["amount"] = display_cols["amount"].map(lambda x: f"{x:.2f}")
    print(display_cols.to_string(index=False))

    if skipped > 0:
        print(f"\n⚠️ Skipped {skipped} invalid/ incomplete row(s) while displaying.")

def set_budget_interactive():
    """Set/update the monthly budget."""
    global budget
    while True:
        try:
            val = input("Enter monthly budget amount (e.g., 500): ").strip()
            budget = float(val)
            if budget < 0:
                print("Budget must be non-negative.")
                continue
            print(f"✅ Budget set to {budget:.2f}")
            return
        except (TypeError, ValueError):
            print("⚠️ Please enter a valid number.")

def track_budget():
    """Compare total spent vs current budget and print remaining/overage."""
    if budget is None:
        print("No budget set yet. Choose option 3 to set one.")
        return
    spent = total_spent(df)
    remaining = round(budget - spent, 2)
    print(f"Total spent so far: {spent:.2f}")
    print(f"Monthly budget:     {budget:.2f}")
    if remaining < 0:
        print(f"🚨 You have exceeded your budget by {-remaining:.2f}!")
    else:
        print(f"✅ You have {remaining:.2f} left for the month.")

# -------------------------------
# 4) FILE I/O (via Pandas)
# -------------------------------
def save_csv(filename=DATA_FILE):
    """Save the cleaned DataFrame to CSV with required columns."""
    f = clean_df(df)
    f.to_csv(filename, index=False)
    print(f"💾 Saved {len(f)} expense(s) to {filename}")

def load_csv(filename=DATA_FILE):
    """Load CSV (if exists) into the global DataFrame."""
    global df
    if not os.path.exists(filename):
        return 0
    try:
        loaded = pd.read_csv(filename, dtype={"date":"string","category":"string","description":"string"})
        # Allow 'amount' coercion later; keep raw read then clean once
        before = len(loaded)
        df = clean_df(loaded)
        after = len(df)
        if before != after:
            print(f"📂 Loaded {after} expense(s) (skipped {before-after} invalid row(s)) from {filename}")
        else:
            print(f"📂 Loaded {after} expense(s) from {filename}")
        return after
    except Exception as e:
        print("⚠️ Could not read CSV:", e)
        df = pd.DataFrame(columns=COLS)
        return 0

# -------------------------------
# 5) MENU LOOP
# -------------------------------
def menu():
    load_csv(DATA_FILE)

    while True:
        print("\n==== Personal Expense Tracker (Pandas) ====")
        print("1) Add expense")
        print("2) View expenses")
        print("3) Track budget (set & compare)")
        print("4) Save expenses")
        print("5) Save & Exit")
        choice = input("Choose an option (1-5): ").strip()

        if choice == "1":
            add_expense_interactive()
        elif choice == "2":
            view_expenses()
        elif choice == "3":
            set_budget_interactive()
            track_budget()
        elif choice == "4":
            save_csv(DATA_FILE)
        elif choice == "5":
            save_csv(DATA_FILE)
            print("👋 Goodbye!")
            break
        else:
            print("⚠️ Invalid choice. Please enter 1-5.")
menu()

ModuleNotFoundError: No module named 'pandas'